Logistic Regression Classification Model
Can an LGA's historical crash profile (2020–2023) predict whether it will be a high-risk LGA in the following year?

Importing all needed libaries like panda and sklearn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.model_selection import  GroupKFold, cross_val_score
from sklearn.inspection import permutation_importance
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
    confusion_matrix, classification_report
)
import joblib, pandas as pd, sys

RANDOM_STATE = 20

Load Dataset

In [ ]:
data = pd.read_csv("../data/processed/lga_yearly_aggregates_clustering.csv")
print(data.shape)
print("LGAs:", data["LGA"].nunique(), "| Years:", sorted(data["Year"].unique()))
print("Missing values:", int(data.isna().sum().sum()))
data.head()

build labels & features

In [ ]:
RATE_COL = "Crash rate per 100k"
FEATURES = ["Population", "Severe %", "Wet-weather %", "Rural-road %"]

train_df = data[data["Year"] <= 2023].copy()
test_df = data[data["Year"] == 2024].copy()

# Threshold (median of crash rate per 100k, 254.9, round down)
risk_threshold = 250.0

train_df["High_risk"] = (train_df[RATE_COL] >= risk_threshold).astype(int)
test_df["High_risk"] = (test_df[RATE_COL] >= risk_threshold).astype(int)

# NumPy arrays with float dtype
X_train = train_df[FEATURES].astype(float).to_numpy()
X_test  = test_df[FEATURES].astype(float).to_numpy()
y_train = train_df["High_risk"].to_numpy()
y_test  = test_df["High_risk"].to_numpy()

# LGA names are only used to keep each LGA inside one cross-validation fold
groups_train = train_df["LGA"].to_numpy()

print(f"High-risk threshold: {risk_threshold:.1f} per 100k")
print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Share of High risk: train {y_train.mean():.1%} | test {y_test.mean():.1%}")

 Logistic Regression & Model Evaluation

In [ ]:
# Initialize and train the Logistic Regression model
log_reg = make_pipeline(
    StandardScaler(),
    LogisticRegression(random_state=RANDOM_STATE, max_iter=1000),
)
log_reg.fit(X_train, y_train)

# Predict on the test data
y_pred_log_reg = log_reg.predict(X_test)
y_prob_log_reg = log_reg.predict_proba(X_test)[:, 1]

# Evaluate Logistic Regression
print("Logistic Regression Evaluation:")
print(f"Accuracy: {accuracy_score(y_test, y_pred_log_reg):.3f}")
print(f"Precision: {precision_score(y_test, y_pred_log_reg):.3f}")
print(f"Recall: {recall_score(y_test, y_pred_log_reg):.3f}")
print(f"F1-Score: {f1_score(y_test, y_pred_log_reg):.3f}")
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_log_reg))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_log_reg))

In [ ]:
# Naive baseline on the same test set
dummy = DummyClassifier(strategy="most_frequent").fit(X_train, y_train)
print(f"Dummy accuracy: {dummy.score(X_test, y_test):.4f}")

# ROC-AUC on the test set
print(f"Test ROC-AUC:   {roc_auc_score(y_test, y_prob_log_reg):.4f}")

# Grouped CV so an LGA's rows never span folds
cv = GroupKFold(n_splits=5)
cv_scores = cross_val_score(
    log_reg, X_train, y_train,
    groups=groups_train, cv=cv,
    scoring="roc_auc",
)
print(f"CV ROC-AUC:     {cv_scores.mean():.3f} ± {cv_scores.std():.3f}")

Save Logistic Regression for visulization

In [ ]:
# Save Logistic Regression for visulization
try:
    perm
except NameError:
    perm = permutation_importance(
        log_reg, X_test, y_test,
        n_repeats=20, random_state=RANDOM_STATE, scoring="roc_auc",
    )

# 1. Per-LGA test predictions
test_out = test_df[["LGA", "Year"]].copy()
test_out["y_true"] = y_test
test_out["y_pred"] = y_pred_log_reg
test_out["y_prob"] = y_prob_log_reg
test_out.to_csv("../data/processed/logreg_test_predictions.csv", index=False)

# 2. Headline metrics (test set + dummy baseline)
metrics = pd.DataFrame([{
    "accuracy":       accuracy_score(y_test, y_pred_log_reg),
    "precision":      precision_score(y_test, y_pred_log_reg),
    "recall":         recall_score(y_test, y_pred_log_reg),
    "f1":             f1_score(y_test, y_pred_log_reg),
    "roc_auc":        roc_auc_score(y_test, y_prob_log_reg),
    "dummy_accuracy": dummy.score(X_test, y_test),
    "cv_roc_auc_mean": cv_scores.mean(),
    "cv_roc_auc_std":  cv_scores.std(),
}])
metrics.to_csv("../data/processed/logreg_metrics.csv", index=False)

# 3. CV fold scores
pd.DataFrame({
    "fold": range(1, len(cv_scores) + 1),
    "cv_roc_auc": cv_scores,
}).to_csv("../data/processed/logreg_cv_scores.csv", index=False)

# 4. Coefficients
pd.DataFrame({
    "feature": FEATURES,
    "coefficient": log_reg.named_steps["logisticregression"].coef_[0],
}).to_csv("../data/processed/logreg_coefficients.csv", index=False)

# 5. Permutation importance
pd.DataFrame({
    "feature": FEATURES,
    "importance_mean": perm.importances_mean,
    "importance_std":  perm.importances_std,
}).to_csv("../data/processed/logreg_permutation_importance.csv", index=False)

joblib.dump({
    "pipeline": log_reg,
    "threshold": 0.5,
    "features": FEATURES,
}, "../models/lga_risk_model.joblib")

print("Saved logistic-regression outputs to processed")